In [ ]:
import smtplib
from email.message import EmailMessage
import bokeh
import os
from pandasgui import show
import pandas as pd
import numpy as np
#Read Excel for prior year term credits
PriorCredits = pd.read_excel(r"C:\Users\iabaker\OneDrive - Boston University\Data Projects\Operations\2026 admissions cycle enrollment updates\Prior year average entering term credits.xlsx"
)
#Rename necessary columns
PriorCredits.rename(columns={"Unnamed: 2": "Graduate Application Academic Program"}, inplace=True)
PriorCredits.rename(columns={"Student Certification": "Graduate Application Full Time Indicator"}, inplace=True)
PriorCredits.rename(columns={"Average Credit Hours": "Average Credits"}, inplace=True)
#Create Online dummy
PriorCredits['Online'] = np.where(
    PriorCredits['Graduate Application Program Campus Location'].str.contains('Online', case=False, na=False),
    'Yes',
    'No'
)
#Create Modality variable
PriorCredits['Modality'] = np.where(
    PriorCredits['Online'] == 'Yes',
    'Online',
    'In person'
)
#Create full time variable
PriorCredits['Graduate Application Full Time Indicator'] = PriorCredits['Graduate Application Full Time Indicator'].map({'Full-Time': 'Y'}).fillna('N')
#Keep only necessary fields
PriorCredits = PriorCredits[[
    'Graduate Application Academic Program',
    'Modality',
    'Graduate Application Full Time Indicator',
    'Average Credits'
]]

In [ ]:
#Read Excel for separated years
PriorYear = pd.read_excel(r"C:\Users\iabaker\OneDrive - Boston University\Data Projects\Operations\2026 admissions cycle enrollment updates\Most Recent ADW Extract_2025 only.xlsx"
)
CurrentYear = pd.read_excel(r"C:\Users\iabaker\OneDrive - Boston University\Data Projects\Operations\2026 admissions cycle enrollment updates\Most Recent ADW Extract_2026 only.xlsx"
)
Data = pd.concat([PriorYear, CurrentYear], ignore_index=True)
# Rename columns
Data.rename(columns={"Unnamed: 2": "TermType"}, inplace=True)
Data.rename(columns={"Unnamed: 3": "TermNumber"}, inplace=True)
Data.rename(columns={"Unnamed: 6": "LastName"}, inplace=True)
Data.rename(columns={"Unnamed: 7": "MiddleName"}, inplace=True)
Data.rename(columns={"Unnamed: 8": "FirstName"}, inplace=True)
Data.rename(columns={"Unnamed: 41": "Dual Application Indicator"}, inplace=True)
#Create Online dummy
Data['Online'] = np.where(
    Data['Graduate Application Program Campus Location'].str.contains('Online', case=False, na=False),
    'Yes',
    'No'
)
#Create Modality variable
Data['Modality'] = np.where(
    Data['Online'] == 'Yes',
    'Online',
    'In person'
)
#Create SU or FA variable
Data['SU or FA'] = np.select(
    [
        Data['Applied To Year Semester'].str.contains("Summer", case=False, na=False),
        Data['Applied To Year Semester'].str.contains("Fall", case=False, na=False)
    ],
    ['SU', 'FA'],
    default=None   # or "" if you prefer blank
)
# Recode Pardee
Data.loc[
    Data['Graduate Application Academic Program'].str.contains('Pardee', na=False),
    'Graduate Application College'
] = 'Pardee School of Global Studies'
#Change Applicant Admitted Date and Graduate Application Tuition Deposit Paid Date to datetime
Data['Applicant Admitted Date'] = pd.to_datetime(
    Data['Applicant Admitted Date'],
    errors='coerce'
)
Data['Graduate Application Tuition Deposit Paid Date'] = pd.to_datetime(
    Data['Graduate Application Tuition Deposit Paid Date'],
    errors='coerce'
)
# Create Referred to Another Program variable
Data['Referred to Another Program'] = np.where(
    Data['Graduate Decision Status'] == 'Referred to Another College',
    'Yes',
    'No'
)
# Create Accepted from Referral variable
Data['Accepted from Referral'] = np.where(
    Data['Graduate Decision Status'] == 'Accepted from Referral',
    'Yes',
    'No'
)
# Recode decision statuses
Data.loc[
    Data['Graduate Decision Status'].str.contains('Accept', na=False),
    'Graduate Decision Status'
] = 'Accept'
# Add a field to sequence degree levels by priority
degree_map = {
    'DOCTORAL DEGREE': 1,
    'PROFESSIONAL DOCTORAL': 2,
    'PROFESSIONAL DOCTORATE': 2,
    'MASTERS DEGREE': 3,
    'POST-DOCTORAL AWARD': 4,
    'POST-MASTERS AWARD': 5,
    'POST-BACHELORS AWARD': 6
}
Data['Degree Level Precedence'] = Data['Graduate Application Degree Level'].map(degree_map)
# Merge w/ prior credits
WCreditsData = pd.merge(
    Data, 
    PriorCredits, 
    on=[
        'Graduate Application Academic Program', 
        'Modality', 
        'Graduate Application Full Time Indicator'
    ], 
    how='left'
)
# Export to Excel
WCreditsData.to_excel(
    r"C:\Users\iabaker\OneDrive - Boston University\Data Projects\Operations\2026 admissions cycle enrollment updates\2025 and 2026 cycle ADW export YTD.xlsx",
    index=False
)

In [ ]:
#Read Excel for combined years
DataCombined = pd.read_excel(r"C:\Users\iabaker\OneDrive - Boston University\Data Projects\Operations\2026 admissions cycle enrollment updates\Most Recent ADW Extract_both years.xlsx"
)
# Rename columns
DataCombined.rename(columns={"Unnamed: 2": "TermType"}, inplace=True)
DataCombined.rename(columns={"Unnamed: 3": "TermNumber"}, inplace=True)
DataCombined.rename(columns={"Unnamed: 6": "LastName"}, inplace=True)
DataCombined.rename(columns={"Unnamed: 7": "MiddleName"}, inplace=True)
DataCombined.rename(columns={"Unnamed: 8": "FirstName"}, inplace=True)
DataCombined.rename(columns={"Unnamed: 40": "Dual Application Indicator"}, inplace=True)
#Create Online dummy
DataCombined['Online'] = np.where(
    DataCombined['Graduate Application Program Campus Location'].str.contains('Online', case=False, na=False),
    'Yes',
    'No'
)
#Create Modality variable
DataCombined['Modality'] = np.where(
    DataCombined['Online'] == 'Yes',
    'Online',
    'In person'
)
#Create SU or FA variable
DataCombined['SU or FA'] = np.select(
    [
        DataCombined['Applied To Year Semester'].str.contains("Summer", case=False, na=False),
        DataCombined['Applied To Year Semester'].str.contains("Fall", case=False, na=False)
    ],
    ['SU', 'FA'],
    default=None   # or "" if you prefer blank
)
# Recode Pardee
DataCombined.loc[
    DataCombined['Graduate Application Academic Program'].str.contains('Pardee', na=False),
    'Graduate Application College'
] = 'Pardee School of Global Studies'
#Change Applicant Admitted Date and Graduate Application Tuition Deposit Paid Date to datetime
DataCombined['Applicant Admitted Date'] = pd.to_datetime(
    DataCombined['Applicant Admitted Date'],
    errors='coerce'
)
DataCombined['Graduate Application Tuition Deposit Paid Date'] = pd.to_datetime(
    DataCombined['Graduate Application Tuition Deposit Paid Date'],
    errors='coerce'
)
# Create Referred to Another Program variable
DataCombined['Referred to Another Program'] = np.where(
    DataCombined['Graduate Decision Status'] == 'Referred to Another College',
    'Yes',
    'No'
)
# Create Accepted from Referral variable
DataCombined['Accepted from Referral'] = np.where(
    DataCombined['Graduate Decision Status'] == 'Accepted from Referral',
    'Yes',
    'No'
)
# Recode decision statuses
DataCombined.loc[
    DataCombined['Graduate Decision Status'].str.contains('Accept', na=False),
    'Graduate Decision Status'
] = 'Accept'
# Add a field to sequence degree levels by priority
degree_map_2 = {
    'DOCTORAL DEGREE': 1,
    'PROFESSIONAL DOCTORAL': 2,
    'PROFESSIONAL DOCTORATE': 2,
    'MASTERS DEGREE': 3,
    'POST-DOCTORAL AWARD': 4,
    'POST-MASTERS AWARD': 5,
    'POST-BACHELORS AWARD': 6
}
DataCombined['Degree Level Precedence'] = DataCombined['Graduate Application Degree Level'].map(degree_map_2)
# Merge w/ estimated credits
WCreditsDataCombined = pd.merge(
    DataCombined, 
    PriorCredits, 
    on=[
        'Graduate Application Academic Program', 
        'Modality', 
        'Graduate Application Full Time Indicator'
    ], 
    how='left'
)
# Export to Excel
WCreditsDataCombined.to_excel(
    r"C:\Users\iabaker\OneDrive - Boston University\Data Projects\Operations\2026 admissions cycle enrollment updates\2025 and 2026 cycle ADW export TOTAL.xlsx",
    index=False
)

In [ ]:
# All multiple applications for separated years
# Start by copying the existing dataset
DMDF=Data.copy()
#Sort by Application Academic Year, then Person Id, then Degree Level Precedence, then Graduate Application Major
DMDF = (DMDF.sort_values(by=['Application Academic Year','Person Id', 'Degree Level Precedence','Graduate Application Major'],
        ascending=[True, True, True, True])
    .reset_index(drop=True))
#Create a variable counting the applications per student per year
DMDF['Application Sequence']= (DMDF.groupby(['Application Academic Year', 'Person Id']).cumcount()+1)
#Define pivot fields to move 2nd through 10th applications into the first applicant-year pairing
pivot_fields = {
    'Graduate Application Major': 'Application Major',
    'Application Received Date': 'Application Received Date', 
    'Graduate Decision Status': 'Graduate Decision Status'}
#Pivot 2nd through 7th dual applications into the first row in a student-year pairing
DMDF_wide = DMDF.copy()
for seq in range(2, 10):
    suffix = (
        f'{seq}nd' if seq == 2 else
        f'{seq}rd' if seq == 3 else
        f'{seq}th'
    )
    seq_df = (
        DMDF
        .loc[DMDF['Application Sequence'] == seq,
             [
                 'Application Academic Year',
                 'Person Id',
                 'Graduate Application Major',
                 'Application Received Date'
             ]]
        .drop_duplicates(
            subset=['Application Academic Year', 'Person Id']
        )
        .rename(columns={
            'Graduate Application Major': f'{suffix} Application Major',
            'Application Received Date': f'{suffix} Application Received Date'
        })
    )
    DMDF_wide = DMDF_wide.merge(
        seq_df,
        on=['Application Academic Year', 'Person Id'],
        how='left'
    )
DMDF_final = (
    DMDF_wide
    .loc[DMDF_wide['Application Sequence'] == 1]
    .reset_index(drop=True)
)
# Export to Excel
DMDF_final.to_excel(
    r"C:\Users\iabaker\OneDrive - Boston University\Data Projects\Operations\2026 admissions cycle enrollment updates\2025 and 2026 cycle multiple applications YTD.xlsx",
    index=False
)

In [ ]:
# Dual degree applications only for separated years
# Start by copying the existing dataset
Duals=Data.copy()
#Keep only observations where Dual Application Indicator = 1
Duals = Duals.loc[Duals['Dual Application Indicator'] == 1]
# Keep only students who have more than 1 application in a given year
Duals = Duals.groupby(['Application Academic Year', 'Person Id']).filter(lambda x: len(x) > 1).copy()
#Sort by Application Academic Year, then Person Id, then Graduate Application Major
Duals = (Duals.sort_values(by=['Application Academic Year','Person Id', 'Degree Level Precedence','Graduate Application Major'],
        ascending=[True, True, True, True])
    .reset_index(drop=True))
#Create a variable counting the applications per student per year
Duals['Application Sequence']= (Duals.groupby(['Application Academic Year', 'Person Id']).cumcount()+1)
#Define pivot fields to move 2nd through 7th applications into the first applicant-year pairing
pivot_fields = {
    'Graduate Application Major': 'Application Major',
    'Application Received Date': 'Application Received Date'}
#Pivot 2nd through 7th dual applications into the first row in a student-year pairing
Duals_wide = Duals.copy()
for seq in range(2, 8):

    suffix = (
        f'{seq}nd' if seq == 2 else
        f'{seq}rd' if seq == 3 else
        f'{seq}th'
    )

    seq_df = (
        Duals
        .loc[Duals['Application Sequence'] == seq,
             [
                 'Application Academic Year',
                 'Person Id',
                 'Graduate Application Major',
                 'Application Received Date'
             ]]
        .drop_duplicates(
            subset=['Application Academic Year', 'Person Id']
        )
        .rename(columns={
            'Graduate Application Major': f'{suffix} Application Major',
            'Application Received Date': f'{suffix} Application Received Date'
        })
    )

    Duals_wide = Duals_wide.merge(
        seq_df,
        on=['Application Academic Year', 'Person Id'],
        how='left'
    )
Duals_final = (
    Duals_wide
    .loc[Duals_wide['Application Sequence'] == 1]
    .reset_index(drop=True)
)
# Export to Excel
Duals_final.to_excel(
    r"C:\Users\iabaker\OneDrive - Boston University\Data Projects\Operations\2026 admissions cycle enrollment updates\2025 and 2026 cycle dual applications YTD.xlsx",
    index=False
)

In [ ]:
# CHINA AND INDIA YIELD EVENT ATTENDEES
# Import Excel spreadsheet of yield event attendees
YieldAttendees = pd.read_excel(
    r"C:\Users\iabaker\OneDrive - Boston University\Data Projects\Operations\2026 admissions cycle enrollment updates\China and India yield events datasets\China and India yield event attendee list_combined.xlsx",
    sheet_name="Data"
)
# Create a dataframe with only current year admits
Admits = Data[
    (Data["Application Academic Year"] == "2026-2027") & 
    (Data["Graduate Decision Status"] == "Accept")
].copy()
# Merge yield event attendees w/ current year admits
YieldAttendeesWAdmitData = pd.merge(
    YieldAttendees, 
    Admits, 
    on="Person Id", 
    how="left",
    indicator="Matched_In_Admits"
)
# Sort resulting dataframe so accepted offers are first by Person Id, then duplicates are dropped, then index is reset
YieldAttendeesWAdmitData = YieldAttendeesWAdmitData.sort_values(
    by=["Person Id", "Graduate Applicant Accepted Response Indicator"], 
    ascending=[True, False]
)
YieldAttendeesWAdmitData = YieldAttendeesWAdmitData.drop_duplicates(subset="Person Id", keep="first")
YieldAttendeesWAdmitData = YieldAttendeesWAdmitData.reset_index(drop=True)
# Export to Excel
YieldAttendeesWAdmitData.to_excel(
    r"C:\Users\iabaker\OneDrive - Boston University\Data Projects\Operations\2026 admissions cycle enrollment updates\China and India yield events datasets\Yield event attendees w accepted offer data.xlsx",
    index=False
)

In [ ]:
# Sort resulting dataframe so accepted offers are first by Person Id, then duplicates are dropped, then index is reset
YieldAttendeesWAdmitData = YieldAttendeesWAdmitData.sort_values(
    by=["Person Id", "Graduate Applicant Accepted Response Indicator"], 
    ascending=[True, False]
)
YieldAttendeesWAdmitData = YieldAttendeesWAdmitData.drop_duplicates(subset="Person Id", keep="first")
YieldAttendeesWAdmitData = YieldAttendeesWAdmitData.reset_index(drop=True)

In [ ]:
# FINAL STEP: SEND AN EMAIL INDICATING THAT ALL STEPS IN THE PYTHON CODE HAVE BEEN RUN CORRECTLY ===
import win32com.client as win32

# Connect to Outlook
outlook = win32.Dispatch('outlook.application')

mail = outlook.CreateItem(0)
mail.To = "iabaker@bu.edu"
mail.Subject = "2026 Supplemental PowerBI data files updated successfully"
mail.Body = (
    "Hello,\n\n"
    "This is an automated notification that the Fall 2026 supplemental admissions data PowerBI data files "
    "have been cleaned, and the cleaned Excel files have been updated successfully.\n\n"
    "Cheers,\n"
    "Ian"
)
mail.Send()

print("Email sent successfully via Outlook client!")